# 06 - HyDe（假设性文档嵌入）

**复杂度：** ⭐⭐⭐

**应用场景：** 模糊查询、领域术语、包含缩写的查询

**核心特性：** 生成假设性的"完美答案"文档，对其进行嵌入，然后用于检索。

**示例：**
```
查询："MMR 是如何工作的？"

假设性文档：
"MMR（最大边际相关性）通过迭代选择与查询相关且与已选文档
不相似的文档，来平衡相关性与多样性..."

→ 嵌入这个详细描述可以找到更好的语义匹配
```

In [1]:
import sys
sys.path.append('../..')

from shared.config import create_chat_model, create_embeddings, DEFAULT_MODEL, DEFAULT_TEMPERATURE, DEFAULT_VISION_MODEL
from shared.config import HF_VECTOR_STORE_PATH, DEFAULT_MODEL
from shared.utils import load_vector_store, print_section_header, format_docs
from shared.prompts import HYDE_PROMPT, RAG_PROMPT_TEMPLATE
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

print_section_header("Setup: HyDe")

embeddings = create_embeddings()
vectorstore = load_vector_store(HF_VECTOR_STORE_PATH, embeddings)
llm = create_chat_model(model=DEFAULT_MODEL, temperature=0)

print("✅ Setup complete!")


SETUP: HYDE



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded vector store from d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\advanced_architectures\..\..\data\vector_stores\huggingface_embeddings


langchain-openai injected a custom httpx transport to apply `http_socket_options`, which disables httpx's proxy auto-detection (system proxy configuration detected). Set `LANGCHAIN_OPENAI_TCP_KEEPALIVE=0` or pass `http_socket_options=()` to restore default proxy behavior, or supply `openai_proxy` / your own `http_client` / `http_async_client` to take full control.


✅ Setup complete!


## 2. HyDe 文档生成器

In [2]:
print_section_header("HyDe Generator")

# Create HyDe document generator
hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

# Test
query = "What is semantic search?"
print(f"Query: '{query}'\n")

hypo_doc = hyde_generator.invoke({"question": query})
print("Generated Hypothetical Document:")
print("=" * 80)
print(hypo_doc)
print("=" * 80)


HYDE GENERATOR

Query: 'What is semantic search?'

Generated Hypothetical Document:
# Semantic Search: A Comprehensive Overview

## Definition

Semantic search refers to a set of search techniques that aim to understand the **intent** and **contextual meaning** of a user's query, rather than relying solely on exact keyword matching. By interpreting the semantic relationship between words, concepts, and entities, semantic search systems can retrieve more relevant results—even when the query does not contain the exact words found in the target documents.

## Core Principles

1. **Intent Understanding**  
   Semantic search goes beyond surface-level terms to infer what the user actually wants. For example, a query like “best way to remove coffee stains from a white shirt” would be understood as a request for cleaning methods, not for coffee or shirt definitions.

2. **Contextual Meaning**  
   The same word can have different meanings depending on context. Semantic search uses surroundin

## 3. HyDe 检索

In [3]:
from shared.utils import print_results

print_section_header("HyDe vs Standard Retrieval")

query = "How to improve retrieval quality?"

# Standard retrieval
print("[STANDARD RETRIEVAL]")
standard_docs = vectorstore.similarity_search(query, k=3)
print_results(standard_docs, max_docs=2, preview_length=120)

# HyDe retrieval
print("\n" + "=" * 80)
print("\n[HYDE RETRIEVAL]")
hypo_doc = hyde_generator.invoke({"question": query})
print(f"\nGenerated doc preview: {hypo_doc[:200]}...\n")
hyde_docs = vectorstore.similarity_search(hypo_doc, k=3)
print_results(hyde_docs, max_docs=2, preview_length=120)

print("\n💡 HyDe often finds more semantically relevant documents")


HYDE VS STANDARD RETRIEVAL

[STANDARD RETRIEVAL]

Retrieved Documents
--------------------------------------------------------------------------------

1. Source: https://python.langchain.com/docs/use_cases/question_answering/
   Type: local_text_download
   Date: 2026-05-09
   Content: {
doc
.
page_content
}
"
)
for
doc
in
retrieved_docs
)
return
serialized
,
retrieved_docs
Here we use the
tool decorator...

2. Source: https://python.langchain.com/docs/use_cases/chatbots/
   Type: local_text_download
   Date: 2026-05-09
   Content: {
doc
.
page_content
}
"
)
for
doc
in
retrieved_docs
)
return
serialized
,
retrieved_docs
Here we use the
tool decorator...

... and 1 more documents


[HYDE RETRIEVAL]

Generated doc preview: # Improving Retrieval Quality in RAG Systems

## Overview

Retrieval quality is a critical factor in the performance of Retrieval-Augmented Generation (RAG) systems. A poor retrieval pipeline can lead...


Retrieved Documents
-----------------------------------------

## 4. HyDe RAG 链

In [4]:
print_section_header("HyDe RAG Chain")

def hyde_retrieve(query: str):
    hypo_doc = hyde_generator.invoke({"question": query})
    docs = vectorstore.similarity_search(hypo_doc, k=4)
    return docs

hyde_retriever = RunnableLambda(hyde_retrieve)

hyde_chain = (
    {"context": hyde_retriever | format_docs, "input": RunnablePassthrough()}
    | RAG_PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)

print("✓ HyDe RAG chain created")

# Test
query = "Best practices for chunk sizing?"
print(f"\nQuery: '{query}'\n")
print("=" * 80)

response = hyde_chain.invoke(query)
print(response)
print("\n" + "=" * 80)


HYDE RAG CHAIN

✓ HyDe RAG chain created

Query: 'Best practices for chunk sizing?'

Based on the provided context, there is no explicit discussion of best practices for chunk sizing. The context only shows a specific example where `chunk_size=1000` characters and `chunk_overlap=200` characters were used, resulting in 66 sub-documents from a blog post. However, it does not explain why those values were chosen or offer general guidelines.

If you need best practices, you would need to consult additional sources (e.g., LangChain documentation or community recommendations) beyond what is available here.



## 总结

**流程：**
```
查询 → 生成假设文档 → 嵌入 → 检索 → LLM → 响应
```

**优势：**
✅ 更适合模糊查询  
✅ 处理专业术语和缩写  
✅ 改善语义匹配  
✅ 适用于专业领域  

**局限性：**
- 额外的 LLM 调用（成本 + 延迟）
- 可能在假设文档中产生幻觉
- 并不总是优于标准方法

**适用场景：**
- 模糊或不明确的查询
- 技术术语
- 包含缩写的查询

**下一步：** [07_adaptive_rag.ipynb](07_adaptive_rag.ipynb) - 智能查询路由